In [2]:
# ==========================================
# BÀI 7: XỬ LÝ TRÙNG LẶP (ĐỘC LẬP)
# ==========================================
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Drive và Đọc dữ liệu từ file Excel
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')
df_clean = df_raw.copy()

# 2. Tiền xử lý kiểu dữ liệu (từ Bài 6) để phục vụ cho Bài 7
df_clean['Created_Date'] = pd.to_datetime(df_clean['Created_Date'], errors='coerce')

print("=== BẮT ĐẦU THỰC HIỆN BÀI 7 ===\n")

# Ghi nhận số dòng ban đầu
initial_rows = len(df_clean)
print(f"Số lượng dòng trước khi xử lý trùng lặp: {initial_rows}")


# ------------------------------------------
# Yêu cầu 31: Loại dòng trùng hoàn toàn trước
# ------------------------------------------
df_clean.drop_duplicates(inplace=True)
after_full_dup_rows = len(df_clean)
removed_full_dup = initial_rows - after_full_dup_rows

print(f"-> Đã loại bỏ {removed_full_dup} dòng trùng lặp hoàn toàn.")
print(f"   Số dòng còn lại sau khi xóa trùng hoàn toàn: {after_full_dup_rows}\n")


# ------------------------------------------
# Yêu cầu 32: Xử lý trùng khóa Product_ID (Nội dung khác nhau)
# ------------------------------------------
# Tạo cột Duplicate_ID_Flag để gắn cờ xác minh cho các mã Product_ID bị trùng lặp thông tin
dup_ids = df_clean[df_clean.duplicated(subset=['Product_ID'], keep=False)]['Product_ID'].unique()

# Gán cờ: True nếu Product_ID bị xung đột dữ liệu, False nếu không
df_clean['Duplicate_ID_Flag'] = df_clean['Product_ID'].isin(dup_ids)

# Quy tắc giữ lại 1 dòng cho mỗi Product_ID theo thứ tự ưu tiên (Sắp xếp Created_Date giảm dần)
df_clean.sort_values(by=['Product_ID', 'Created_Date'], ascending=[True, False], inplace=True)
df_clean.drop_duplicates(subset=['Product_ID'], keep='first', inplace=True)
df_clean.reset_index(drop=True, inplace=True)

after_key_dup_rows = len(df_clean)


# ------------------------------------------
# Yêu cầu 33: Báo cáo số dòng trước và sau xử lý
# ------------------------------------------
print("--- 33. BÁO CÁO TỔNG HỢP SỐ DÒNG XỬ LÝ TRÙNG LẶP ---")
print(f"1. Số dòng ban đầu (trước xử lý): {initial_rows}")
print(f"2. Số dòng trùng hoàn toàn đã xóa: {removed_full_dup}")
print(f"3. Số dòng trùng khóa Product_ID đã xử lý: {after_full_dup_rows - after_key_dup_rows}")
print(f"4. Số dòng còn lại sau cùng: {after_key_dup_rows}")

print("\n--- ĐỀ XUẤT QUY TẮC VÀ HẠN CHẾ ---")
print("- Quy tắc chọn dòng ưu tiên: Ưu tiên giữ lại bản ghi có Created_Date mới nhất. Nếu ngày tạo giống/khuyết, giữ lại dòng xuất hiện đầu tiên.")
print("- Cờ đánh dấu: Tạo cột 'Duplicate_ID_Flag = True' cho các sản phẩm từng xuất hiện xung đột dữ liệu để phục vụ kiểm tra lại.")
print("- Hạn chế: Do dữ liệu không có trường 'Updated_Time', việc lọc theo Created_Date hoặc vị trí dòng có thể chưa phản ánh chính xác trạng thái mới nhất của sản phẩm.")

Mounted at /content/drive
=== BẮT ĐẦU THỰC HIỆN BÀI 7 ===

Số lượng dòng trước khi xử lý trùng lặp: 228
-> Đã loại bỏ 5 dòng trùng lặp hoàn toàn.
   Số dòng còn lại sau khi xóa trùng hoàn toàn: 223

--- 33. BÁO CÁO TỔNG HỢP SỐ DÒNG XỬ LÝ TRÙNG LẶP ---
1. Số dòng ban đầu (trước xử lý): 228
2. Số dòng trùng hoàn toàn đã xóa: 5
3. Số dòng trùng khóa Product_ID đã xử lý: 3
4. Số dòng còn lại sau cùng: 220

--- ĐỀ XUẤT QUY TẮC VÀ HẠN CHẾ ---
- Quy tắc chọn dòng ưu tiên: Ưu tiên giữ lại bản ghi có Created_Date mới nhất. Nếu ngày tạo giống/khuyết, giữ lại dòng xuất hiện đầu tiên.
- Cờ đánh dấu: Tạo cột 'Duplicate_ID_Flag = True' cho các sản phẩm từng xuất hiện xung đột dữ liệu để phục vụ kiểm tra lại.
- Hạn chế: Do dữ liệu không có trường 'Updated_Time', việc lọc theo Created_Date hoặc vị trí dòng có thể chưa phản ánh chính xác trạng thái mới nhất của sản phẩm.
